In [0]:
from pyspark.sql.types import StringType, IntegerType, DateType, BooleanType, TimestampType
import pyspark.sql.functions as F
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("catalog_name", "ecommerce", "Catalog Name")
dbutils.widgets.text("storage_account_name", "stgecommerceeastus001", "Storage Account Name")
dbutils.widgets.text("container_name", "ecomm-raw-data", "Container Name")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
storage_account_name = dbutils.widgets.get("storage_account_name")
container_name = dbutils.widgets.get("container_name")

In [0]:
df = spark.readStream \
    .format('delta') \
    .option('readChangeFeed', 'true') \
    .table(f"{catalog_name}.silver.slv_order_returns")



In [0]:
df_union = df.filter("_change_type IN ('insert', 'update_postimage')")

In [0]:
orders_returns_gold_df = df_union.withColumn(
    'date_id',
    F.date_format('order_dt', 'yyyyMMdd').cast('int')
).withColumn(
    'return_days',
    F.date_diff(F.col('order_dt'), F.col('return_ts'))
).withColumn(
    'within_policy',
    F.when(
        F.col('return_days') <= 15, 1).otherwise(0)
).withColumn(
    'is_late_return',
    F.when(
        F.col('return_days') > 15, 1).otherwise(0)
)


In [0]:
gold_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoint/gold/fact_order_returns/"
print(gold_checkpoint_path)

In [0]:
def upsert_to_gold(microBatchDF, batchId):
    table_name = f"{catalog_name}.gold.gld_fact_order_returns"
    spark.sql(f"USE CATALOG {catalog_name}")
    spark.sql("USE SCHEMA gold")
    microBatchDF = microBatchDF.drop("_change_type", "_commit_version", "_commit_timestamp")
    if not spark.catalog.tableExists(table_name):
        print("creating new table")
        microBatchDF.write.format("delta").mode("overwrite").saveAsTable(table_name)
        spark.sql(
            f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
        )
    else:
        microBatchDF.createOrReplaceTempView("batch_table")
        spark.sql(f"""
            MERGE INTO {table_name} AS gold_table
            USING batch_table
            ON gold_table.primary_key = batch_table.primary_key
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)

orders_returns_gold_df.writeStream.trigger(availableNow=True).foreachBatch(
    upsert_to_gold
).option("checkpointLocation", gold_checkpoint_path).option(
    "mergeSchema", "true"
).outputMode(
    "update"
).start().awaitTermination()